# Lab 2: Build RNN Models (LSTM, GRU, Bidirectional)

In this lab, we build and compare four different Recurrent Neural Network architectures for sentiment classification on the IMDB dataset.

## Objectives

- Understand RNN fundamentals and LSTM/GRU gate mechanisms
- Build and train four RNN variants: LSTM, GRU, Bidirectional LSTM, and Stacked LSTM
- Compare validation accuracy and training time across all four models
- Explore how different architectures affect performance

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")

## 1. RNN Fundamentals

### Why RNNs?

Standard feedforward networks treat each input independently. For sequences (text, time series, audio), we need models that can capture **temporal dependencies** -- the meaning of a word depends on the words that came before it.

### The Vanishing Gradient Problem

Simple RNNs struggle with long sequences because gradients either vanish (shrink to zero) or explode (grow unboundedly) during backpropagation through time.

### LSTM (Long Short-Term Memory)

LSTMs solve this with three gates:

| Gate | Purpose |
|------|---------|
| **Forget gate** | Decides what information to discard from the cell state |
| **Input gate** | Decides what new information to store in the cell state |
| **Output gate** | Decides what to output based on the cell state |

### GRU (Gated Recurrent Unit)

GRUs simplify the LSTM by combining the forget and input gates into a single **update gate** and merging the cell state and hidden state. Fewer parameters, often similar performance.

### Bidirectional RNNs

Process the sequence in both forward and backward directions, then concatenate the outputs. This captures context from both past and future tokens.

### Stacked RNNs

Multiple RNN layers stacked on top of each other. Intermediate layers must use `return_sequences=True` to pass full sequence outputs to the next layer.

## 2. Load and Prepare the IMDB Dataset

In [ ]:
# Load IMDB dataset and decode to text
(x_train_enc, y_train), (x_test_enc, y_test) = keras.datasets.imdb.load_data()

word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = "<pad>"
reverse_word_index[1] = "<start>"
reverse_word_index[2] = "<unk>"
reverse_word_index[3] = "<unused>"

def decode_review(encoded_review):
    return " ".join(reverse_word_index.get(i, "?") for i in encoded_review)

x_train_text = np.array([decode_review(seq) for seq in x_train_enc])
x_test_text = np.array([decode_review(seq) for seq in x_test_enc])

print(f"Training samples: {len(x_train_text)}")
print(f"Test samples: {len(x_test_text)}")

In [ ]:
# Create a shared TextVectorization layer
MAX_TOKENS = 10000
MAX_LENGTH = 200
EMBEDDING_DIM = 128

text_vectorizer = keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_sequence_length=MAX_LENGTH,
    output_mode="int",
)
text_vectorizer.adapt(x_train_text)

# Pre-vectorize the data so we can reuse it across models
x_train_vec = text_vectorizer(x_train_text).numpy()
x_test_vec = text_vectorizer(x_test_text).numpy()

print(f"Vectorized shape: {x_train_vec.shape}")

## 3. Build Four RNN Models

We define a helper function to build each model variant, then train and evaluate them.

In [ ]:
def build_model_a():
    """Model A: Embedding -> LSTM(64) -> Dense(1, sigmoid)"""
    model = keras.Sequential([
        keras.layers.Input(shape=(MAX_LENGTH,)),
        keras.layers.Embedding(MAX_TOKENS, EMBEDDING_DIM),
        keras.layers.LSTM(64),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    return model

def build_model_b():
    """Model B: Embedding -> GRU(64) -> Dense(1, sigmoid)"""
    model = keras.Sequential([
        keras.layers.Input(shape=(MAX_LENGTH,)),
        keras.layers.Embedding(MAX_TOKENS, EMBEDDING_DIM),
        keras.layers.GRU(64),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    return model

def build_model_c():
    """Model C: Embedding -> Bidirectional(LSTM(64)) -> Dense(1, sigmoid)"""
    model = keras.Sequential([
        keras.layers.Input(shape=(MAX_LENGTH,)),
        keras.layers.Embedding(MAX_TOKENS, EMBEDDING_DIM),
        keras.layers.Bidirectional(keras.layers.LSTM(64)),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    return model

def build_model_d():
    """Model D: Embedding -> LSTM(64, return_sequences=True) -> LSTM(32) -> Dense(1, sigmoid)"""
    model = keras.Sequential([
        keras.layers.Input(shape=(MAX_LENGTH,)),
        keras.layers.Embedding(MAX_TOKENS, EMBEDDING_DIM),
        keras.layers.LSTM(64, return_sequences=True),
        keras.layers.LSTM(32),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    return model

model_builders = {
    "A: LSTM": build_model_a,
    "B: GRU": build_model_b,
    "C: Bidirectional LSTM": build_model_c,
    "D: Stacked LSTM": build_model_d,
}

# Show summaries
for name, builder in model_builders.items():
    print(f"\n{'='*60}")
    print(f"Model {name}")
    print(f"{'='*60}")
    m = builder()
    m.summary()

## 4. Train All Four Models

In [ ]:
EPOCHS = 5
BATCH_SIZE = 64

results = {}
trained_models = {}

for name, builder in model_builders.items():
    print(f"\n{'='*60}")
    print(f"Training Model {name}")
    print(f"{'='*60}")

    model = builder()
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    start_time = time.time()
    history = model.fit(
        x_train_vec, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        verbose=1,
    )
    training_time = time.time() - start_time

    test_loss, test_acc = model.evaluate(x_test_vec, y_test, verbose=0)

    results[name] = {
        "val_accuracy": max(history.history["val_accuracy"]),
        "test_accuracy": test_acc,
        "training_time": training_time,
        "history": history.history,
    }
    trained_models[name] = model

    print(f"\nTest accuracy: {test_acc:.4f} | Time: {training_time:.1f}s")

## 5. Compare Results

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame({
    "Model": results.keys(),
    "Best Val Accuracy": [r["val_accuracy"] for r in results.values()],
    "Test Accuracy": [r["test_accuracy"] for r in results.values()],
    "Training Time (s)": [r["training_time"] for r in results.values()],
}).set_index("Model")

comparison_df = comparison_df.round(4)
print(comparison_df.to_string())

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Validation accuracy curves
for name, res in results.items():
    axes[0].plot(res["history"]["val_accuracy"], label=name)
axes[0].set_title("Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend(fontsize=8)

# Validation loss curves
for name, res in results.items():
    axes[1].plot(res["history"]["val_loss"], label=name)
axes[1].set_title("Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend(fontsize=8)

# Bar chart: test accuracy vs training time
model_names = list(results.keys())
test_accs = [results[n]["test_accuracy"] for n in model_names]
bar_colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
axes[2].bar(range(len(model_names)), test_accs, color=bar_colors)
axes[2].set_xticks(range(len(model_names)))
axes[2].set_xticklabels([n.split(":")[0] for n in model_names])
axes[2].set_title("Test Accuracy")
axes[2].set_ylim(0.7, 0.95)

plt.tight_layout()
plt.show()